# How to clean and normalise text

Real-world text is messy. It arrives with inconsistent spacing, mixed cases, accented characters, and unpredictable line endings. Before you can process text reliably, you need to clean and normalise it into a consistent form.

This guide walks through the most common text cleaning techniques using Python's built-in string methods and the `unicodedata` module. By the end, you will have a reusable `clean_text()` function that combines all of these techniques.

## Stripping whitespace

Leading and trailing whitespace is one of the most common sources of bugs when comparing or storing text. Python provides three methods for removing it:

- `str.strip()` removes whitespace from both ends
- `str.lstrip()` removes whitespace from the left (start) only
- `str.rstrip()` removes whitespace from the right (end) only

In [ ]:
raw_input = "   Hello, world!   \n"

print(repr(raw_input))
print(repr(raw_input.strip()))
print(repr(raw_input.lstrip()))
print(repr(raw_input.rstrip()))

You can also strip specific characters by passing a string argument. The method removes any combination of those characters from the relevant end.

In [ ]:
data = "***important***"

print(data.strip("*"))       # Removes asterisks from both ends
print(data.lstrip("*"))      # Removes asterisks from the left only
print(data.rstrip("*"))      # Removes asterisks from the right only

## Normalising case

Case normalisation is essential for case-insensitive comparisons, searching, and deduplication. Python offers two approaches:

- `str.lower()` converts to lowercase using simple rules
- `str.casefold()` performs aggressive lowercase folding, which handles special characters from other languages more thoroughly

For most English text, `str.lower()` is sufficient. Use `str.casefold()` when you need to handle international text correctly.

In [ ]:
english_text = "Hello World"
german_text = "Straße"  # German word containing the sharp s character

print(english_text.lower())       # "hello world"
print(english_text.casefold())    # "hello world" (same result for English)

print(german_text.lower())        # "straße" (sharp s is unchanged)
print(german_text.casefold())     # "strasse" (sharp s is expanded to "ss")

In [ ]:
# Demonstration: why casefold() matters for comparisons
word_a = "Straße"
word_b = "STRASSE"

print(word_a.lower() == word_b.lower())        # False (lower does not expand sharp s)
print(word_a.casefold() == word_b.casefold())   # True (casefold handles it correctly)

## Removing accents

Sometimes you need to convert accented characters to their plain ASCII equivalents, for example when creating URL slugs or performing simplified searches. The `unicodedata` module provides the tools for this.

The technique works in two steps:

1. Decompose the string into base characters and combining marks using `unicodedata.normalize("NFD", ...)`
2. Filter out the combining marks (Unicode category `"Mn"`) to leave only the base characters

In [ ]:
import unicodedata


def remove_accents(text: str) -> str:
    """Remove accent marks from characters, converting them to plain ASCII equivalents."""
    # NFD decomposes characters: for example, 'é' becomes 'e' + combining acute accent
    normalised = unicodedata.normalize("NFD", text)
    # Filter out combining marks (category "Mn" means Mark, Nonspacing)
    return "".join(
        char for char in normalised
        if unicodedata.category(char) != "Mn"
    )


print(remove_accents("café"))          # "cafe"
print(remove_accents("naïve"))         # "naive"
print(remove_accents("résumé"))        # "resume"
print(remove_accents("Ångström"))      # "Angstrom"

Note that this approach works well for Latin-based scripts but does not handle all Unicode transformations. Characters such as the German sharp s (`ß`) or ligatures require separate handling.

## Collapsing multiple spaces

Text copied from documents, web pages, or user input often contains multiple consecutive spaces. A simple and effective way to collapse them into single spaces is to split and rejoin the string.

When called without arguments, `str.split()` splits on any whitespace (spaces, tabs, newlines) and discards empty strings from the result. Joining the result with a single space produces clean, uniformly spaced text.

In [ ]:
messy_text = "  This   has    too     many   spaces  "

# Split on whitespace, then rejoin with a single space
clean = " ".join(messy_text.split())

print(repr(messy_text))
print(repr(clean))

In [ ]:
# This also handles tabs and newlines
mixed_whitespace = "Name:\tJohn\n\nAge:  25"

print(repr(mixed_whitespace))
print(repr(" ".join(mixed_whitespace.split())))

## Normalising line endings

Different operating systems use different line ending conventions:

- Unix and macOS use `\n` (line feed)
- Windows uses `\r\n` (carriage return followed by line feed)
- Classic Mac OS used `\r` (carriage return only)

When processing text from multiple sources, it is good practice to normalise all line endings to a single convention. The standard approach is to convert everything to `\n`.

In [ ]:
def normalise_line_endings(text: str) -> str:
    """Convert all line endings to Unix-style line feed characters."""
    # Replace Windows line endings first (order matters)
    text = text.replace("\r\n", "\n")
    # Then replace any remaining carriage returns
    text = text.replace("\r", "\n")
    return text


mixed_endings = "Line one\r\nLine two\rLine three\nLine four"

print(repr(mixed_endings))
print(repr(normalise_line_endings(mixed_endings)))

## Putting it all together: a complete `clean_text()` function

The following function combines all of the techniques above into a single reusable function. You can enable or disable individual cleaning steps depending on your requirements.

In [ ]:
import unicodedata


def clean_text(
    text: str,
    strip_whitespace: bool = True,
    normalise_case: bool = True,
    remove_accent_marks: bool = False,
    collapse_spaces: bool = True,
    normalise_endings: bool = True,
) -> str:
    """Clean and normalise text using a combination of techniques.

    Args:
        text: The text to clean.
        strip_whitespace: Remove leading and trailing whitespace.
        normalise_case: Convert text to lowercase using casefold.
        remove_accent_marks: Remove accent marks from characters.
        collapse_spaces: Replace multiple spaces with a single space.
        normalise_endings: Convert all line endings to Unix-style.

    Returns:
        The cleaned and normalised text.
    """
    if normalise_endings:
        text = text.replace("\r\n", "\n").replace("\r", "\n")

    if strip_whitespace:
        text = text.strip()

    if normalise_case:
        text = text.casefold()

    if remove_accent_marks:
        normalised = unicodedata.normalize("NFD", text)
        text = "".join(
            char for char in normalised
            if unicodedata.category(char) != "Mn"
        )

    if collapse_spaces:
        # Preserve line breaks by processing each line separately
        lines = text.split("\n")
        lines = [" ".join(line.split()) for line in lines]
        text = "\n".join(lines)

    return text

In [ ]:
# Test the clean_text() function with messy input
messy = "  Héllo,   WORLD!  \r\n  This is a   résumé.  \r"

print("Original:")
print(repr(messy))
print()

print("Cleaned (default settings):")
print(repr(clean_text(messy)))
print()

print("Cleaned (with accent removal):")
print(repr(clean_text(messy, remove_accent_marks=True)))
print()

print("Cleaned (case preserved):")
print(repr(clean_text(messy, normalise_case=False)))

## Alternative approaches

The techniques in this guide rely entirely on Python's built-in capabilities. For more advanced text cleaning needs, you may wish to consider the following third-party libraries:

- **`unidecode`** provides more comprehensive transliteration of Unicode characters to ASCII, handling a wider range of scripts and special characters than the `unicodedata` approach shown above.
- **`ftfy`** ("fixes text for you") repairs common encoding issues such as mojibake, where text has been decoded with the wrong character encoding.
- **`regex`** (the `regex` module on PyPI) offers more powerful regular expression features than the built-in `re` module, including better Unicode support.

For most standard text cleaning tasks, however, the built-in methods shown in this guide are sufficient and have the advantage of requiring no additional dependencies.

## Summary

In this guide, you learned how to:

- Strip leading and trailing whitespace with `str.strip()`, `str.lstrip()`, and `str.rstrip()`
- Normalise case with `str.lower()` and `str.casefold()`
- Remove accent marks using `unicodedata.normalize()` and `unicodedata.category()`
- Collapse multiple spaces into one with `" ".join(text.split())`
- Normalise line endings by replacing `\r\n` and `\r` with `\n`
- Combine all techniques into a reusable `clean_text()` function